# 🌿 LeafScan — Bitki Hastalığı Tespiti (Plant Disease Detection)

Uçtan uca pipeline: **veri hazırlama → klasik makine öğrenmesi (feature engineering + classifiers) → derin öğrenme (transfer learning + fine-tuning, v1→v6) → değerlendirme (evaluation)**.

> Bu notebook, iki ayrı çalışmanın (klasik yöntem + CNN denemeleri) temizlenip tek dosyada, mantıksal sıraya konmuş halidir. Tekrar eden ve başarısız deneme hücreleri çıkarıldı. Google Colab'da yukarıdan aşağıya çalıştırılacak şekilde düzenlenmiştir.

## 0. Kurulum ve Veri İndirme
Kaggle API anahtarını (kaggle.json) yükle ve PlantVillage veri setini indir + aç.

In [ ]:
from google.colab import files
files.upload()  # kaggle.json'ı seç

In [ ]:
import os
os.makedirs("/root/.kaggle", exist_ok=True)
os.rename("kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 600)

# Dataseti indir
os.system("kaggle datasets download -d abdallahalidev/plantvillage-dataset")

# Zip'i aç
import zipfile
with zipfile.ZipFile("plantvillage-dataset.zip", "r") as z:
    z.extractall("plantvillage")

print("Tamamlandı!")

## 1. Veri Keşfi (EDA)
Sınıf sayısı/dağılımına bak ve farklı sınıflardan örnek yaprak görsellerini göster.

In [ ]:
import os

base_dir = "plantvillage/plantvillage dataset/color"

siniflar = os.listdir(base_dir)
print(f"Toplam sınıf sayısı: {len(siniflar)}")
print("\nİlk 10 sınıf:")
for s in siniflar[:10]:
    n = len(os.listdir(os.path.join(base_dir, s)))
    print(f"  {s}: {n} görüntü")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import random, os

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
axes = axes.flatten()

# 8 farklı sınıftan rastgele birer görüntü göster
secilen = random.sample(siniflar, 8)

for i, sinif in enumerate(secilen):
    sinif_yolu = os.path.join(base_dir, sinif)
    goruntu = random.choice(os.listdir(sinif_yolu))
    img = mpimg.imread(os.path.join(sinif_yolu, goruntu))
    axes[i].imshow(img)
    axes[i].set_title(sinif.replace("___", "\n"), fontsize=7)
    axes[i].axis("off")

plt.tight_layout()
plt.show()

## 2. Klasik Yöntem — Öznitelik Çıkarımı + Sınıflandırıcılar
Örüntü tanımanın klasik yaklaşımı: her görselden **renk histogramı + ortalama/std + HOG** öznitelikleri çıkarılır (feature extraction), ardından **SelectKBest** ile en iyi öznitelikler seçilip **Random Forest** ve **SVM** ile sınıflandırılır.

In [ ]:
import cv2
import numpy as np
from skimage.feature import hog
from sklearn.preprocessing import LabelEncoder
import os

def goruntu_ozellik_cikart(img_path):
    # Görüntüyü oku ve yeniden boyutlandır
    img = cv2.imread(img_path)
    img = cv2.resize(img, (128, 128))

    # 1. Renk histogramı (RGB her kanal için)
    hist_r = np.histogram(img[:,:,0], bins=32, range=(0,256))[0]
    hist_g = np.histogram(img[:,:,1], bins=32, range=(0,256))[0]
    hist_b = np.histogram(img[:,:,2], bins=32, range=(0,256))[0]

    # 2. Ortalama ve standart sapma
    ortalama = img.mean(axis=(0,1))
    std = img.std(axis=(0,1))

    # 3. HOG özellikleri
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    hog_feat = hog(gray, orientations=8, pixels_per_cell=(16,16),
                   cells_per_block=(1,1), feature_vector=True)

    # Hepsini birleştir
    return np.concatenate([hist_r, hist_g, hist_b, ortalama, std, hog_feat])

# Veriyi yükle (her sınıftan max 200 görüntü al - hız için)
X, y = [], []
for sinif in siniflar:
    sinif_yolu = os.path.join(base_dir, sinif)
    goruntular = os.listdir(sinif_yolu)[:200]
    for g in goruntular:
        try:
            ozellik = goruntu_ozellik_cikart(os.path.join(sinif_yolu, g))
            X.append(ozellik)
            y.append(sinif)
        except:
            pass

X = np.array(X)
y = np.array(y)

print(f"Toplam örnek: {len(X)}")
print(f"Her örneğin özellik sayısı: {X.shape[1]}")
print(f"Sınıf sayısı: {len(set(y))}")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
import pickle

# Etiketleri sayıya çevir
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Eğitim / test olarak böl (%80 eğitim, %20 test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# Feature selection — en iyi 100 özelliği seç
selector = SelectKBest(f_classif, k=100)
X_train_sel = selector.fit_transform(X_train, y_train)
X_test_sel = selector.transform(X_test)

print(f"Seçilen özellik sayısı: {X_train_sel.shape[1]}")

# Model eğitimi
print("Model eğitiliyor...")
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train_sel, y_train)

# Test
y_pred = model.predict(X_test_sel)
acc = accuracy_score(y_test, y_pred)
print(f"\nModel doğruluğu: %{acc*100:.2f}")

# Modeli kaydet
with open("model.pkl", "wb") as f:
    pickle.dump(model, f)
with open("selector.pkl", "wb") as f:
    pickle.dump(selector, f)
with open("label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

print("Model kaydedildi!")

In [ ]:
# Daha güçlü model deneyelim
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.svm import SVC

# 1. Random Forest ama daha fazla ağaç ile
model2 = RandomForestClassifier(
    n_estimators=200,      # 100 yerine 200 ağaç
    max_depth=None,
    min_samples_split=2,
    random_state=42,
    n_jobs=-1
)
model2.fit(X_train_sel, y_train)
y_pred2 = model2.predict(X_test_sel)
acc2 = accuracy_score(y_test, y_pred2)
print(f"Random Forest (200 ağaç): %{acc2*100:.2f}")

# 2. SVM dene
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_sel)
X_test_scaled = scaler.transform(X_test_sel)

svm = SVC(kernel='rbf', C=10, random_state=42)
svm.fit(X_train_scaled, y_train)
y_pred_svm = svm.predict(X_test_scaled)
acc_svm = accuracy_score(y_test, y_pred_svm)
print(f"SVM: %{acc_svm*100:.2f}")

# En iyisini kaydet
best_acc = max(acc2, acc_svm)
print(f"\nEn iyi doğruluk: %{best_acc*100:.2f}")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import pickle

# SVM'i kaydet
with open("model.pkl", "wb") as f:
    pickle.dump(svm, f)
with open("scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

print("SVM modeli kaydedildi!")

# Confusion matrix
cm = confusion_matrix(y_test, y_pred_svm)
fig, ax = plt.subplots(figsize=(16, 14))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
disp.plot(ax=ax, xticks_rotation=90, colorbar=False, cmap="Greens")
ax.set_title("Confusion Matrix — Bitki Hastalığı Tespiti", fontsize=14)
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("Confusion matrix kaydedildi!")

In [ ]:
# En iyi 20 tahmini gösteren bar chart
from sklearn.metrics import classification_report
import pandas as pd

# Sınıf bazında başarı oranları
report = classification_report(y_test, y_pred_svm,
                                target_names=le.classes_,
                                output_dict=True)
df_report = pd.DataFrame(report).T
df_sinif = df_report[:-3]  # summary satırlarını çıkar

# En iyi 10 ve en kötü 10 sınıfı göster
df_sinif = df_sinif.sort_values("f1-score", ascending=False)

fig, ax = plt.subplots(figsize=(12, 8))
renkler = ["#2d8c4e" if f >= 0.7 else "#e07b39" for f in df_sinif["f1-score"]]
bars = ax.barh(df_sinif.index, df_sinif["f1-score"], color=renkler)
ax.set_xlabel("F1 Score", fontsize=12)
ax.set_title("Sınıf Bazında F1 Score — Bitki Hastalığı Tespiti", fontsize=14)
ax.axvline(x=0.7, color="gray", linestyle="--", alpha=0.5, label="%70 eşik")
ax.legend()
ax.set_xlim(0, 1)
plt.tight_layout()
plt.savefig("f1_scores.png", dpi=150, bbox_inches="tight")
plt.show()

# Genel özet yazdır
print(f"\nGenel Accuracy: %{accuracy_score(y_test, y_pred_svm)*100:.2f}")
print(f"Ortalama F1: {df_sinif['f1-score'].mean():.2f}")
print(f"\nEn iyi 3 sınıf:")
print(df_sinif['f1-score'].head(3))
print(f"\nEn zor 3 sınıf:")
print(df_sinif['f1-score'].tail(3))

In [ ]:
from google.colab import files

# 3 dosyayı indir
files.download("model.pkl")
files.download("selector.pkl")
files.download("label_encoder.pkl")
files.download("scaler.pkl")
files.download("confusion_matrix.png")
files.download("f1_scores.png")

print("Hepsi indirildi!")

## 3. Derin Öğrenme — Taban Model (v1)
**MobileNetV2** (transfer learning) tabanı **donuk** (frozen) tutulur; üstüne özel sınıflandırma katmanları (GlobalAveragePooling + Dropout + Dense) eklenir ve eğitilir.

→ `best_model.keras`

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os

# GPU var mı kontrol et
print("GPU:", tf.config.list_physical_devices('GPU'))

base_dir = "plantvillage/plantvillage dataset/color"

# Veri hazırlığı
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=20,
    horizontal_flip=True,
    zoom_range=0.2
)

train_gen = datagen.flow_from_directory(
    base_dir,
    target_size=(128, 128),
    batch_size=32,
    subset='training',
    class_mode='categorical'
)

val_gen = datagen.flow_from_directory(
    base_dir,
    target_size=(128, 128),
    batch_size=32,
    subset='validation',
    class_mode='categorical'
)

print(f"Eğitim: {train_gen.samples} görüntü")
print(f"Test: {val_gen.samples} görüntü")
print(f"Sınıf sayısı: {len(train_gen.class_indices)}")

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

# Hazır MobileNetV2 modelini yükle
base_model = MobileNetV2(
    input_shape=(128, 128, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # hazır ağırlıkları dondur

# Üstüne bizim katmanlarımızı ekle
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
x = Dense(128, activation='relu')(x)
output = Dense(38, activation='softmax')(x)

model_cnn = Model(inputs=base_model.input, outputs=output)

model_cnn.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Model hazır!")
model_cnn.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import json

early_stop = EarlyStopping(patience=3, restore_best_weights=True)
checkpoint = ModelCheckpoint("best_model.keras", save_best_only=True)

history = model_cnn.fit(
    train_gen,
    epochs=10,
    validation_data=val_gen,
    callbacks=[early_stop, checkpoint]
)

# Sınıf isimlerini kaydet
with open("class_names.json", "w") as f:
    json.dump(train_gen.class_indices, f)

print("Eğitim tamamlandı!")

## 4. Güçlü Augmentation ile Yeniden Eğitim (v2)
Daha agresif veri artırma (rotation 40°, yatay+dikey flip, brightness, zoom) ile model **sıfırdan**, daha uzun (15 epoch) eğitilir.

→ `best_model_v2.keras`

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print("GPU:", tf.config.list_physical_devices('GPU'))

base_dir = "plantvillage/plantvillage dataset/color"

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=40,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.3,
    brightness_range=[0.7, 1.3],
    shear_range=0.2,
    fill_mode='nearest'
)

train_gen = datagen.flow_from_directory(
    base_dir, target_size=(128,128),
    batch_size=32, subset='training',
    class_mode='categorical'
)

val_gen = datagen.flow_from_directory(
    base_dir, target_size=(128,128),
    batch_size=32, subset='validation',
    class_mode='categorical'
)

print(f"Eğitim: {train_gen.samples} görüntü")
print(f"Test: {val_gen.samples} görüntü")

In [ ]:
# v2 ayrı bir eğitimdir: daha güçlü augmentation ile modeli SIFIRDAN yeniden kuruyoruz
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

base_model = MobileNetV2(input_shape=(128,128,3), include_top=False, weights='imagenet')
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
x = Dense(128, activation='relu')(x)
output = Dense(38, activation='softmax')(x)

model_cnn = Model(inputs=base_model.input, outputs=output)
model_cnn.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
print("Model hazır!")

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import json

early_stop = EarlyStopping(patience=3, restore_best_weights=True)
checkpoint = ModelCheckpoint("best_model_v2.keras", save_best_only=True)

history = model_cnn.fit(
    train_gen,
    epochs=15,
    validation_data=val_gen,
    callbacks=[early_stop, checkpoint]
)

with open("class_names.json", "w") as f:
    json.dump(train_gen.class_indices, f)

print("Eğitim tamamlandı!")

## 5. Fine-Tuning (v3)
v2 ağırlıkları yüklenir, MobileNetV2'nin **son 30 katmanı** açılıp düşük learning rate (1e-4) ile ince ayar (fine-tuning) yapılır.

→ `best_model_v3.keras`

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Önceki eğitilmiş ağırlıkları yükle
model_cnn.load_weights("best_model_v2.keras")
print("Önceki ağırlıklar yüklendi!")

# Son 30 katmanı aç
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model_cnn.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(patience=3, restore_best_weights=True)
checkpoint = ModelCheckpoint("best_model_v3.keras", save_best_only=True)

history_ft = model_cnn.fit(
    train_gen,
    epochs=10,
    validation_data=val_gen,
    callbacks=[early_stop, checkpoint]
)

print(f"Fine-tuning tamamlandı!")
print(f"En iyi val_accuracy: {max(history_ft.history['val_accuracy'])*100:.2f}%")

## 6. Devam Eğitimi (v4)
v3 üzerinden eğitim sürdürülür ve sınıf isimleri `class_names.json` olarak kaydedilir.

→ `best_model_v4.keras`

In [ ]:
from tensorflow.keras.models import load_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import json

model_cnn = load_model("best_model_v3.keras")
print("Model yüklendi!")

model_cnn.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(patience=3, restore_best_weights=True)
checkpoint = ModelCheckpoint("best_model_v4.keras", save_best_only=True)

history = model_cnn.fit(
    train_gen, epochs=10,
    validation_data=val_gen,
    callbacks=[early_stop, checkpoint]
)

with open("class_names.json", "w") as f:
    json.dump(train_gen.class_indices, f)

print(f"Tamamlandı! En iyi: {max(history.history['val_accuracy'])*100:.2f}%")

## 7. Derin Fine-Tuning (v5)
v4 yüklenir, **son 50 katman** açılır ve çok düşük learning rate (1e-5) ile ince ayar yapılır. (~%98.00 val accuracy)

→ `best_model_v5.keras`

In [ ]:
import tensorflow as tf
from tensorflow import keras

# Önceki bölümde kaydedilen v4 modelini yükle
model = keras.models.load_model("best_model_v4.keras", compile=False)
print("Model yüklendi!")
print(f"Input shape: {model.input_shape}")
print(f"Output shape: {model.output_shape}")

In [ ]:
import json

with open("class_names.json", "r") as f:
    class_indices = json.load(f)

print(f"Sınıf sayısı: {len(class_indices)}")
print(list(class_indices.items())[:5])

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Augmentation ile data generator
datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

# PlantVillage train
train_gen = datagen.flow_from_directory(
    plantvillage_path,
    target_size=(128, 128),
    batch_size=32,
    class_mode='categorical',
    subset='training',
    classes=list(class_indices.keys())
)

# PlantVillage validation
val_gen = datagen.flow_from_directory(
    plantvillage_path,
    target_size=(128, 128),
    batch_size=32,
    class_mode='categorical',
    subset='validation',
    classes=list(class_indices.keys())
)

print(f"Train: {train_gen.samples} görsel")
print(f"Validation: {val_gen.samples} görsel")

In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Son 50 katmanı aç
model.trainable = True
for layer in model.layers[:-50]:
    layer.trainable = False

print(f"Eğitilebilir katman: {sum(1 for l in model.layers if l.trainable)}")

# Derle
model.compile(
    optimizer=Adam(learning_rate=0.00001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Callbacks
early_stop = EarlyStopping(patience=3, restore_best_weights=True)
checkpoint = ModelCheckpoint("best_model_v5.keras", save_best_only=True)

# Eğit
history = model.fit(
    train_gen,
    epochs=10,
    validation_data=val_gen,
    callbacks=[early_stop, checkpoint]
)

print("Fine-tune tamamlandı!")
print(f"En iyi val_accuracy: {max(history.history['val_accuracy'])*100:.2f}%")

In [ ]:
from google.colab import files
files.download("best_model_v5.keras")

## 8. PlantDoc Entegrasyonu + Son Model (v6)
Gerçek dünya yaprak fotoğrafları içeren **PlantDoc** indirilir, sınıfları PlantVillage sınıflarına **eşlenip** kopyalanır; model en düşük learning rate (5e-6) ile son kez fine-tune edilir. **(~%98.34 — aktif model)**

→ `best_model_v6.keras`

In [ ]:
os.system("kaggle datasets download -d nirmalsankalana/plantdoc-dataset")
print("İndirildi!")

In [ ]:
import zipfile
with zipfile.ZipFile("plantdoc-dataset.zip", "r") as z:
    z.extractall("plantdoc")
print("PlantDoc hazır!")

# Klasör yapısına bak
import os
for f in os.listdir("plantdoc")[:10]:
    print(f)

In [ ]:
# Tüm PlantDoc klasörlerini görelim
print(sorted(os.listdir(plantdoc_path)))

In [ ]:
# PlantDoc → PlantVillage eşleştirme
plantdoc_mapping = {
    'Apple_Scab_Leaf': 'Apple___Apple_scab',
    'Apple_leaf': 'Apple___healthy',
    'Apple_rust_leaf': 'Apple___Cedar_apple_rust',
    'Bell_pepper_leaf': 'Pepper,_bell___healthy',
    'Bell_pepper_leaf_spot': 'Pepper,_bell___Bacterial_spot',
    'Blueberry_leaf': 'Blueberry___healthy',
    'Cherry_leaf': 'Cherry_(including_sour)___healthy',
    'Corn_Gray_leaf_spot': 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot',
    'Corn_leaf_blight': 'Corn_(maize)___Northern_Leaf_Blight',
    'Corn_rust_leaf': 'Corn_(maize)___Common_rust_',
    'Peach_leaf': 'Peach___healthy',
    'Potato_leaf_early_blight': 'Potato___Early_blight',
    'Potato_leaf_late_blight': 'Potato___Late_blight',
    'Raspberry_leaf': 'Raspberry___healthy',
    'Soyabean_leaf': 'Soybean___healthy',
    'Squash_Powdery_mildew_leaf': 'Squash___Powdery_mildew',
    'Strawberry_leaf': 'Strawberry___healthy',
    'Tomato_Early_blight_leaf': 'Tomato___Early_blight',
    'Tomato_Septoria_leaf_spot': 'Tomato___Septoria_leaf_spot',
    'Tomato_leaf': 'Tomato___healthy',
    'Tomato_leaf_bacterial_spot': 'Tomato___Bacterial_spot',
    'Tomato_leaf_late_blight': 'Tomato___Late_blight',
    'Tomato_leaf_mosaic_virus': 'Tomato___Tomato_mosaic_virus',
    'Tomato_leaf_yellow_virus': 'Tomato___Tomato_Yellow_Leaf_Curl_Virus',
    'Tomato_mold_leaf': 'Tomato___Leaf_Mold',
    'Tomato_two_spotted_spider_mites_leaf': 'Tomato___Spider_mites Two-spotted_spider_mite',
    'grape_leaf': 'Grape___healthy',
    'grape_leaf_black_rot': 'Grape___Black_rot',
}

print(f"Eşleştirilen sınıf: {len(plantdoc_mapping)}")

In [ ]:
import shutil

copied = 0
for plantdoc_class, plantvillage_class in plantdoc_mapping.items():
    src = os.path.join(plantdoc_path, plantdoc_class)
    dst = os.path.join(plantvillage_path, plantvillage_class)

    if os.path.exists(src) and os.path.exists(dst):
        for img in os.listdir(src):
            shutil.copy(
                os.path.join(src, img),
                os.path.join(dst, img)
            )
            copied += 1

print(f"Kopyalanan görsel: {copied}")

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

train_gen = datagen.flow_from_directory(
    plantvillage_path,
    target_size=(128, 128),
    batch_size=32,
    class_mode='categorical',
    subset='training',
    classes=list(class_indices.keys())
)

val_gen = datagen.flow_from_directory(
    plantvillage_path,
    target_size=(128, 128),
    batch_size=32,
    class_mode='categorical',
    subset='validation',
    classes=list(class_indices.keys())
)

print(f"Train: {train_gen.samples} görsel")
print(f"Validation: {val_gen.samples} görsel")

In [ ]:
# Modeli yeniden yükle (v5'ten devam edelim)
model = keras.models.load_model("best_model_v5.keras", compile=False)

# Son 50 katmanı aç
model.trainable = True
for layer in model.layers[:-50]:
    layer.trainable = False

print(f"Eğitilebilir katman: {sum(1 for l in model.layers if l.trainable)}")

model.compile(
    optimizer=Adam(learning_rate=0.000005),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(patience=3, restore_best_weights=True)
checkpoint = ModelCheckpoint("best_model_v6.keras", save_best_only=True)

history = model.fit(
    train_gen,
    epochs=10,
    validation_data=val_gen,
    callbacks=[early_stop, checkpoint]
)

print("Fine-tune tamamlandı!")
print(f"En iyi val_accuracy: {max(history.history['val_accuracy'])*100:.2f}%")

In [ ]:
from google.colab import files
files.download("best_model_v6.keras")

## 9. Değerlendirme (Evaluation)
`evaluate_models.py` ile v6 modelinin metrikleri (accuracy, precision, recall, F1-score, ROC vb.) hesaplanır ve `metrics_cache.json` olarak kaydedilir.

> Not: `evaluate_models.py` proje deposundan gelir; aşağıdaki hücrede önce onu yüklemen gerekir.

In [ ]:
from google.colab import files

# evaluate_models.py proje deposundan gelir — çalıştırmadan önce yükle
uploaded = files.upload()  # evaluate_models.py seç
print("evaluate_models.py yüklendi!")

In [ ]:
# Dosyayı oku ve yolu değiştir
with open('evaluate_models.py', 'r') as f:
    content = f.read()

# Yolları düzelt
content = content.replace(
    "'/content/../bitki_projesi_model/class_names.json'",
    "'class_names.json'"
).replace(
    '../bitki_projesi_model/',
    ''
).replace(
    'bitki_projesi_model/',
    ''
)

with open('evaluate_models.py', 'w') as f:
    f.write(content)

print("Yollar düzeltildi!")

In [ ]:
# Dosyayı oku
with open('evaluate_models.py', 'r') as f:
    content = f.read()

# CLASS_NAMES_PATH satırını bul ve değiştir
import re
content = re.sub(
    r"CLASS_NAMES_PATH\s*=.*",
    "CLASS_NAMES_PATH = 'class_names.json'",
    content
)
content = re.sub(
    r"MODEL_DIR\s*=.*",
    "MODEL_DIR = '.'",
    content
)

with open('evaluate_models.py', 'w') as f:
    f.write(content)

print("Düzeltildi!")

# Kontrol et
with open('evaluate_models.py', 'r') as f:
    for line in f:
        if 'CLASS_NAMES_PATH' in line or 'MODEL_DIR' in line:
            print(line.strip())

In [ ]:
with open('evaluate_models.py', 'r') as f:
    content = f.read()

content = content.replace(
    "MODEL_NAMES = ['best_model', 'best_model_v2', 'best_model_v3', 'best_model_v4']",
    "MODEL_NAMES = ['best_model_v6']"
)

with open('evaluate_models.py', 'w') as f:
    f.write(content)

print("Düzeltildi!")

In [ ]:
import subprocess
result = subprocess.run(
    ['python', 'evaluate_models.py', '--quick'],
    capture_output=True, text=True
)
print(result.stdout[-3000:])
print(result.stderr[-2000:])

In [ ]:
from google.colab import files
files.download('metrics_cache.json')